In [11]:
# ==========================================================
# MLP EXPERIMENTS – QUESTION 6
# Single Cell Implementation (Compatible with your mlp.py)
# ==========================================================

import numpy as np
import matplotlib.pyplot as plt
import time

from mlp import MLP
import sys
sys.path.insert(0, '../ADALINE_Adaptive_Linear_Neuron')
from adaline import Adaline


# ==========================================================
# LOAD DATASET
# ==========================================================

print("\n===== Loading Dataset =====")

data = np.loadtxt("../IIT_H_Campus_Life_Prediction_Dataset/iit_h_mess_dataset.csv", delimiter=",", skiprows=1)

X = data[:, :-1]
y = data[:, -1].reshape(-1,1)

n = len(X)

indices = np.random.permutation(n)
X = X[indices]
y = y[indices]

train_end = int(0.7*n)
val_end = int(0.85*n)

X_train = X[:train_end]
y_train = y[:train_end]

X_val = X[train_end:val_end]
y_val = y[train_end:val_end]

X_test = X[val_end:]
y_test = y[val_end:]

print("Train:",X_train.shape)
print("Validation:",X_val.shape)
print("Test:",X_test.shape)



# ==========================================================
# HELPER FUNCTION
# ==========================================================

def plot_history(history,title):

    plt.figure()

    plt.plot(history)

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)

    plt.show()



# ==========================================================
# DEPTH ABLATION
# ==========================================================

print("\n===== Depth Ablation =====")

depth_archs = [
    [X.shape[1],64,1],
    [X.shape[1],64,64,1],
    [X.shape[1],64,64,64,1],
    [X.shape[1],64,64,64,64,1]
]

depth_acc = []

for arch in depth_archs:

    activations = ['relu']*(len(arch)-2) + ['sigmoid']

    model = MLP(
        layer_sizes=arch,
        activations=activations,
        optimizer='adam',
        weight_init='he'
    )

    history = model.fit(X_train,y_train,epochs=50)

    pred = model.predict(X_test)

    acc = np.mean((pred>0.5)==y_test)

    depth_acc.append(acc)

    plot_history(history,f"Depth {len(arch)-2}")


plt.figure()
plt.plot([1,2,3,4],depth_acc)
plt.xlabel("Network Depth")
plt.ylabel("Test Accuracy")
plt.title("Depth vs Accuracy")
plt.show()



# ==========================================================
# WIDTH ABLATION
# ==========================================================

print("\n===== Width Ablation =====")

widths = [8,16,32,64,128,256]

width_acc = []
num_params = []

for w in widths:

    arch = [X.shape[1],w,w,1]

    model = MLP(
        layer_sizes=arch,
        activations=['relu','relu','sigmoid'],
        optimizer='adam'
    )

    history = model.fit(X_train,y_train,epochs=50)

    pred = model.predict(X_test)

    acc = np.mean((pred>0.5)==y_test)

    width_acc.append(acc)

    params = 0
    for i in range(len(arch)-1):
        params += arch[i]*arch[i+1]

    num_params.append(params)


plt.figure()

plt.semilogx(num_params,width_acc)

plt.xlabel("Parameters (log scale)")
plt.ylabel("Test Accuracy")

plt.title("Width vs Accuracy")

plt.show()



# ==========================================================
# ACTIVATION FUNCTION COMPARISON
# ==========================================================

print("\n===== Activation Comparison =====")

configs = {

"sigmoid": ['sigmoid','sigmoid','sigmoid'],
"tanh": ['tanh','tanh','tanh'],
"relu": ['relu','relu','sigmoid'],
"leaky_relu": ['leaky_relu','leaky_relu','sigmoid']

}

results = {}

for name,acts in configs.items():

    model = MLP(
        layer_sizes=[X.shape[1],64,64,1],
        activations=acts,
        optimizer='adam'
    )

    history = model.fit(X_train,y_train,epochs=50)

    pred = model.predict(X_test)

    acc = np.mean((pred>0.5)==y_test)

    results[name] = acc

    plot_history(history,name)

print("\nActivation Results:",results)



# ==========================================================
# OPTIMIZER COMPARISON
# ==========================================================

print("\n===== Optimizer Comparison =====")

optimizers = [
    'sgd',
    'momentum',
    'adam',
    'adagrad',
    'rmsprop',
    'nesterov',
    'muon'
]

opt_results = {}

for opt in optimizers:

    model = MLP(
        layer_sizes=[X.shape[1],64,64,1],
        activations=['relu','relu','sigmoid'],
        optimizer=opt
    )

    history = model.fit(X_train,y_train,epochs=50)

    pred = model.predict(X_test)

    acc = np.mean((pred>0.5)==y_test)

    opt_results[opt] = acc

    plot_history(history,opt)

print("\nOptimizer Results:",opt_results)



# ==========================================================
# REGULARIZATION
# ==========================================================

print("\n===== Regularization =====")

lambdas = [0.001,0.01,0.1]

for lam in lambdas:

    model = MLP(
        layer_sizes=[X.shape[1],64,64,1],
        activations=['relu','relu','sigmoid'],
        regularization='l2',
        lambda_reg=lam
    )

    history = model.fit(X_train,y_train,epochs=50)

    plot_history(history,f"L2 lambda={lam}")



# ==========================================================
# PCA IMPLEMENTATION (NUMPY)
# ==========================================================

print("\n===== PCA Feature Visualization =====")

def PCA_2D(X):

    X_mean = np.mean(X,axis=0)
    X_centered = X - X_mean

    cov = np.cov(X_centered,rowvar=False)

    eigvals,eigvecs = np.linalg.eig(cov)

    idx = np.argsort(eigvals)[::-1]

    eigvecs = eigvecs[:,idx]

    W = eigvecs[:,:2]

    return np.dot(X_centered,W)


def extract_features(model,X):

    activations,_ = model.forward(X)

    return activations[-2]


features = extract_features(model,X_test)

proj = PCA_2D(features)

plt.figure()

plt.scatter(proj[:,0],proj[:,1],c=y_test)

plt.title("PCA Feature Visualization")

plt.show()



# ==========================================================
# ADALINE COMPARISON
# ==========================================================

print("\n===== Adaline vs MLP =====")

adaline = Adaline()

adaline.fit(X_train,y_train)

adaline_mse = adaline.score(X_test,y_test)

mlp_pred = model.predict(X_test)

mlp_mse = np.mean((mlp_pred-y_test)**2)

print("Adaline MSE:",adaline_mse)
print("MLP MSE:",mlp_mse)


===== Loading Dataset =====
Train: (3500, 14)
Validation: (750, 14)
Test: (750, 14)

===== Depth Ablation =====


AttributeError: 'str' object has no attribute 'forward'